In [ ]:
!pip install mediapipe opencv-python pandas scipy tqdm numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 9.1 MB/s eta 0:00:00

In [ ]:
# =============================================================================
# SINGLE FILE: DATA PROCESSING & AUGMENTATION (METHOD 3)
# =============================================================================

import cv2
import numpy as np
import os
import mediapipe as mp
import pandas as pd
from tqdm import tqdm
import json
from datetime import datetime
from scipy.interpolate import interp1d
import random
import math

# --- 1. CONFIGURATION ---
# Đường dẫn trên Google Drive (Cập nhật theo đúng máy của bạn)
DRIVE_PATH = '/content/drive/MyDrive/VSL_Data/Phuong/'
DATASET_PATH = os.path.join(DRIVE_PATH, 'Dataset')
OUTPUT_DATA_PATH = os.path.join(DRIVE_PATH, 'Processed_Data_Augmented_V2') # Folder mới cho giai đoạn mới
LOG_PATH = os.path.join(DRIVE_PATH, 'Logs')

# Tạo thư mục nếu chưa có
os.makedirs(OUTPUT_DATA_PATH, exist_ok=True)
os.makedirs(LOG_PATH, exist_ok=True)

# Cấu hình Mediapipe
mp_holistic = mp.solutions.holistic
N_UPPER_BODY_POSE_LANDMARKS = 25
N_HAND_LANDMARKS = 21
N_TOTAL_LANDMARKS = N_UPPER_BODY_POSE_LANDMARKS + N_HAND_LANDMARKS + N_HAND_LANDMARKS # 67 điểm
SEQUENCE_LENGTH = 60 # Số frame cố định đưa vào model

# Cấu hình Augmentation
NUM_AUGMENTS_PER_VIDEO = 100 # SỐ LƯỢNG AUGMENT MUỐN TẠO RA TỪ 1 VIDEO (Càng nhiều càng tốt như bạn yêu cầu)
MAX_AUGS_COMBINED = 4        # Số lượng kĩ thuật kết hợp tối đa trên 1 mẫu (ví dụ: vừa xoay, vừa noise, vừa scale)

# Các chỉ số Landmark quan trọng cho IK
POSE_LM_LEFT_SHOULDER = 11
POSE_LM_RIGHT_SHOULDER = 12
POSE_LM_LEFT_ELBOW = 13
POSE_LM_RIGHT_ELBOW = 14
POSE_LM_LEFT_WRIST = 15
POSE_LM_RIGHT_WRIST = 16

# =============================================================================
# 2. MEDIAPIPE & EXTRACTION UTILS
# =============================================================================

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def extract_keypoints(results):
    pose_kps = np.zeros((N_UPPER_BODY_POSE_LANDMARKS, 3))
    left_hand_kps = np.zeros((N_HAND_LANDMARKS, 3))
    right_hand_kps = np.zeros((N_HAND_LANDMARKS, 3))

    if results and results.pose_landmarks:
        for i in range(N_UPPER_BODY_POSE_LANDMARKS):
            if i < len(results.pose_landmarks.landmark):
                res = results.pose_landmarks.landmark[i]
                pose_kps[i] = [res.x, res.y, res.z]

    if results and results.left_hand_landmarks:
        left_hand_kps = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark])

    if results and results.right_hand_landmarks:
        right_hand_kps = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark])

    keypoints = np.concatenate([pose_kps, left_hand_kps, right_hand_kps])
    return keypoints.flatten()

def sequence_frames(video_path, holistic):
    sequence_frames = []
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total_frames // 100)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % step != 0: continue

        try:
            image, results = mediapipe_detection(frame, holistic)
            keypoints = extract_keypoints(results)
            if keypoints is not None:
                sequence_frames.append(keypoints)
        except Exception:
            continue
    cap.release()
    return sequence_frames

# =============================================================================
# 3. INVERSE KINEMATICS (IK) SOLVER (Code gốc của bạn)
# =============================================================================

def solve_2_link_ik_2d_v2(p_shoulder_xy, p_wrist_target_xy, len_upper_arm, len_forearm, original_elbow_xy=None, original_wrist_xy=None, prefer_original_bend=True):
    d = np.linalg.norm(p_wrist_target_xy - p_shoulder_xy)
    l1 = max(1e-5, len_upper_arm)
    l2 = max(1e-5, len_forearm)

    if d > l1 + l2 - 1e-5:
        if d < 1e-9: return p_shoulder_xy + np.array([l1, 0]) if original_elbow_xy is None else original_elbow_xy.copy()
        vec_sw = (p_wrist_target_xy - p_shoulder_xy) / d
        return p_shoulder_xy + vec_sw * l1

    if d < abs(l1 - l2) + 1e-5:
        if original_elbow_xy is not None: return original_elbow_xy.copy()
        if d < 1e-9: return p_shoulder_xy + np.array([l1, 0])
        vec_sw = (p_wrist_target_xy - p_shoulder_xy) / d
        return p_shoulder_xy + vec_sw * l1

    if d < 1e-9: d = 1e-9
    a = (l1**2 - l2**2 + d**2) / (2 * d)
    h_squared = l1**2 - a**2
    if h_squared < -1e-9:
        vec_sw = (p_wrist_target_xy - p_shoulder_xy) / d
        return p_shoulder_xy + vec_sw * l1
    h = np.sqrt(max(0, h_squared))

    p2_x = p_shoulder_xy[0] + a * (p_wrist_target_xy[0] - p_shoulder_xy[0]) / d
    p2_y = p_shoulder_xy[1] + a * (p_wrist_target_xy[1] - p_shoulder_xy[1]) / d
    perp_vec_x = -(p_wrist_target_xy[1] - p_shoulder_xy[1]) / d
    perp_vec_y =  (p_wrist_target_xy[0] - p_shoulder_xy[0]) / d

    elbow_sol1_xy = np.array([p2_x + h * perp_vec_x, p2_y + h * perp_vec_y])
    elbow_sol2_xy = np.array([p2_x - h * perp_vec_x, p2_y - h * perp_vec_y])

    if not prefer_original_bend or original_elbow_xy is None or original_wrist_xy is None:
        if original_elbow_xy is not None:
            dist1 = np.linalg.norm(elbow_sol1_xy - original_elbow_xy)
            dist2 = np.linalg.norm(elbow_sol2_xy - original_elbow_xy)
            return elbow_sol1_xy if dist1 <= dist2 else elbow_sol2_xy
        return elbow_sol1_xy

    vec_sw_orig = original_wrist_xy - p_shoulder_xy
    if np.linalg.norm(vec_sw_orig) < 1e-5:
        dist1 = np.linalg.norm(elbow_sol1_xy - original_elbow_xy)
        dist2 = np.linalg.norm(elbow_sol2_xy - original_elbow_xy)
        return elbow_sol1_xy if dist1 <= dist2 else elbow_sol2_xy

    original_side = (original_wrist_xy[0] - p_shoulder_xy[0]) * (original_elbow_xy[1] - p_shoulder_xy[1]) - \
                    (original_wrist_xy[1] - p_shoulder_xy[1]) * (original_elbow_xy[0] - p_shoulder_xy[0])
    side1 = (p_wrist_target_xy[0] - p_shoulder_xy[0]) * (elbow_sol1_xy[1] - p_shoulder_xy[1]) - \
            (p_wrist_target_xy[1] - p_shoulder_xy[1]) * (elbow_sol1_xy[0] - p_shoulder_xy[0])
    side2 = (p_wrist_target_xy[0] - p_shoulder_xy[0]) * (elbow_sol2_xy[1] - p_shoulder_xy[1]) - \
            (p_wrist_target_xy[1] - p_shoulder_xy[1]) * (elbow_sol2_xy[0] - p_shoulder_xy[0])

    if abs(original_side) < 1e-3:
        dist1 = np.linalg.norm(elbow_sol1_xy - original_elbow_xy)
        dist2 = np.linalg.norm(elbow_sol2_xy - original_elbow_xy)
        return elbow_sol1_xy if dist1 <= dist2 else elbow_sol2_xy

    if np.sign(side1) == np.sign(original_side): return elbow_sol1_xy
    elif np.sign(side2) == np.sign(original_side): return elbow_sol2_xy
    else:
        dist1 = np.linalg.norm(elbow_sol1_xy - original_elbow_xy)
        dist2 = np.linalg.norm(elbow_sol2_xy - original_elbow_xy)
        return elbow_sol1_xy if dist1 <= dist2 else elbow_sol2_xy

# =============================================================================
# 4. AUGMENTATION FUNCTIONS (Geometric & Noise)
# =============================================================================

def scale_keypoints_sequence(keypoints_sequence, scale_factor_range=(0.7, 1.26)):
    processed_sequence = []
    if not keypoints_sequence: return processed_sequence
    current_scale_factor = random.uniform(scale_factor_range[0], scale_factor_range[1])

    for frame in keypoints_sequence:
        if frame is None:
            processed_sequence.append(None)
            continue
        try:
            points = frame.copy().reshape(N_TOTAL_LANDMARKS, 3)
            # Tính tâm
            valid_mask = np.any(points != 0, axis=1)
            if np.any(valid_mask):
                center_x = np.median(points[valid_mask, 0])
                center_y = np.median(points[valid_mask, 1])

                points[valid_mask, 0] = (points[valid_mask, 0] - center_x) * current_scale_factor + center_x
                points[valid_mask, 1] = (points[valid_mask, 1] - center_y) * current_scale_factor + center_y
            processed_sequence.append(points.flatten())
        except:
            processed_sequence.append(frame)
    return processed_sequence

def rotate_keypoints_sequence(keypoints_sequence, angle_degrees_range=(-15.0, 15.0)):
    processed_sequence = []
    if not keypoints_sequence: return processed_sequence
    angle_deg = random.uniform(angle_degrees_range[0], angle_degrees_range[1])
    angle_rad = math.radians(angle_deg)
    cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)

    for frame in keypoints_sequence:
        if frame is None:
            processed_sequence.append(None)
            continue
        try:
            points = frame.copy().reshape(N_TOTAL_LANDMARKS, 3)
            valid_mask = np.any(points != 0, axis=1)
            if np.any(valid_mask):
                center_x = np.median(points[valid_mask, 0])
                center_y = np.median(points[valid_mask, 1])

                x = points[valid_mask, 0] - center_x
                y = points[valid_mask, 1] - center_y

                points[valid_mask, 0] = x * cos_a - y * sin_a + center_x
                points[valid_mask, 1] = x * sin_a + y * cos_a + center_y
            processed_sequence.append(points.flatten())
        except:
            processed_sequence.append(frame)
    return processed_sequence

def translate_keypoints_sequence(keypoints_sequence, translate_range=(-0.05, 0.05)):
    processed_sequence = []
    dx = random.uniform(*translate_range)
    dy = random.uniform(*translate_range)
    for frame in keypoints_sequence:
        if frame is None:
            processed_sequence.append(None)
            continue
        try:
            points = frame.copy().reshape(N_TOTAL_LANDMARKS, 3)
            valid_mask = np.any(points != 0, axis=1)
            points[valid_mask, 0] += dx
            points[valid_mask, 1] += dy
            processed_sequence.append(points.flatten())
        except:
            processed_sequence.append(frame)
    return processed_sequence

def time_stretch_keypoints_sequence(keypoints_sequence, speed_range=(0.8, 1.2)):
    if not keypoints_sequence: return keypoints_sequence
    valid_frames = [kp for kp in keypoints_sequence if kp is not None]
    if not valid_frames: return keypoints_sequence

    speed_factor = random.uniform(*speed_range)
    original_len = len(valid_frames)
    target_len = int(original_len / speed_factor)
    if target_len <= 0: target_len = 1

    indices = np.linspace(0, original_len - 1, target_len)
    indices = np.round(indices).astype(int)
    indices = np.clip(indices, 0, original_len - 1)

    return [valid_frames[i].copy() for i in indices]

def inter_hand_distance(keypoints_sequence, total_dx_range=(-0.1, 0.1), dy_shift_range=(-0.03, 0.03)):
    processed_sequence = []
    dx_change = random.uniform(*total_dx_range)
    dy_shift = random.uniform(*dy_shift_range)

    for frame in keypoints_sequence:
        if frame is None:
            processed_sequence.append(None)
            continue
        try:
            points = frame.copy().reshape(N_TOTAL_LANDMARKS, 3)
            # (Phần logic IK giữ nguyên như code bạn đã cung cấp, gọi hàm solve_2_link_ik_2d_v2)
            # Để ngắn gọn, tôi tích hợp logic gọi hàm IK ở đây:

            s_l = points[POSE_LM_LEFT_SHOULDER, :2]
            e_l = points[POSE_LM_LEFT_ELBOW, :2]
            w_l = points[POSE_LM_LEFT_WRIST, :2]
            s_r = points[POSE_LM_RIGHT_SHOULDER, :2]
            e_r = points[POSE_LM_RIGHT_ELBOW, :2]
            w_r = points[POSE_LM_RIGHT_WRIST, :2]

            valid_l = np.all(s_l!=0) and np.all(e_l!=0) and np.all(w_l!=0)
            valid_r = np.all(s_r!=0) and np.all(e_r!=0) and np.all(w_r!=0)

            if valid_l and valid_r:
                mid_x = (w_l[0] + w_r[0]) / 2
                dist_x = abs(w_r[0] - w_l[0])
                target_dist = max(0.01, dist_x + dx_change)

                if w_l[0] <= w_r[0]:
                    w_l_target = np.array([mid_x - target_dist/2, w_l[1]])
                    w_r_target = np.array([mid_x + target_dist/2, w_r[1]])
                else:
                    w_r_target = np.array([mid_x - target_dist/2, w_r[1]])
                    w_l_target = np.array([mid_x + target_dist/2, w_l[1]])

                # Solve IK Left
                len_l1 = np.linalg.norm(e_l - s_l)
                len_l2 = np.linalg.norm(w_l - e_l)
                e_l_new = solve_2_link_ik_2d_v2(s_l, w_l_target, len_l1, len_l2, e_l, w_l)
                if e_l_new is not None:
                    d_hand_l = w_l_target - w_l
                    points[POSE_LM_LEFT_ELBOW, :2] = e_l_new
                    points[POSE_LM_LEFT_WRIST, :2] = w_l_target
                    # Dịch chuyển cả bàn tay
                    idx_start = N_UPPER_BODY_POSE_LANDMARKS
                    idx_end = idx_start + N_HAND_LANDMARKS
                    mask_h = np.any(points[idx_start:idx_end] != 0, axis=1)
                    points[idx_start:idx_end][mask_h, 0] += d_hand_l[0]
                    points[idx_start:idx_end][mask_h, 1] += d_hand_l[1]

                # Solve IK Right
                len_r1 = np.linalg.norm(e_r - s_r)
                len_r2 = np.linalg.norm(w_r - e_r)
                e_r_new = solve_2_link_ik_2d_v2(s_r, w_r_target, len_r1, len_r2, e_r, w_r)
                if e_r_new is not None:
                    d_hand_r = w_r_target - w_r
                    points[POSE_LM_RIGHT_ELBOW, :2] = e_r_new
                    points[POSE_LM_RIGHT_WRIST, :2] = w_r_target
                    idx_start = N_UPPER_BODY_POSE_LANDMARKS + N_HAND_LANDMARKS
                    idx_end = idx_start + N_HAND_LANDMARKS
                    mask_h = np.any(points[idx_start:idx_end] != 0, axis=1)
                    points[idx_start:idx_end][mask_h, 0] += d_hand_r[0]
                    points[idx_start:idx_end][mask_h, 1] += d_hand_r[1]

            # Shift Y
            if abs(dy_shift) > 1e-5:
                # Dịch chuyển toàn bộ tay
                valid_idxs = list(range(POSE_LM_LEFT_SHOULDER, N_TOTAL_LANDMARKS))
                mask = np.any(points[valid_idxs, :2] != 0, axis=1)
                points[np.array(valid_idxs)[mask], 1] += dy_shift

            processed_sequence.append(points.flatten())
        except:
             processed_sequence.append(frame)
    return processed_sequence

# --- NEW AUGMENTATIONS ---

def add_gaussian_noise(keypoints_sequence, noise_sigma=0.005):
    processed = []
    for frame in keypoints_sequence:
        if frame is None:
            processed.append(None)
            continue
        noise = np.random.normal(0, noise_sigma, frame.shape)
        # Chỉ add noise vào các điểm khác 0
        mask = frame != 0
        noisy_frame = frame.copy()
        noisy_frame[mask] += noise[mask]
        processed.append(noisy_frame)
    return processed

def stretch_compress_keypoints(keypoints_sequence, range_val=(0.85, 1.15)):
    sx = random.uniform(*range_val)
    sy = random.uniform(*range_val)
    processed = []
    for frame in keypoints_sequence:
        if frame is None: processed.append(None); continue
        try:
            pts = frame.copy().reshape(-1, 3)
            mask = np.any(pts!=0, axis=1)
            if np.any(mask):
                cx = np.median(pts[mask, 0])
                cy = np.median(pts[mask, 1])
                pts[mask, 0] = (pts[mask, 0] - cx) * sx + cx
                pts[mask, 1] = (pts[mask, 1] - cy) * sy + cy
            processed.append(pts.flatten())
        except: processed.append(frame)
    return processed

def random_dropout(keypoints_sequence, drop_prob=0.05):
    processed = []
    for frame in keypoints_sequence:
        if frame is None: processed.append(None); continue
        mask = np.random.choice([0, 1], size=frame.shape, p=[drop_prob, 1-drop_prob])
        processed.append(frame * mask)
    return processed

# =============================================================================
# 5. PRE-PROCESSING & NORMALIZATION PIPELINE
# =============================================================================

def interpolate_and_normalize(keypoints_sequence, target_len=SEQUENCE_LENGTH):
    """
    1. Interpolate về target_len (60).
    2. Normalize Spatial về [0, 1] và Centerize.
    """
    if not keypoints_sequence: return None
    # Loại bỏ None frames
    valid_seq = [x for x in keypoints_sequence if x is not None]
    if not valid_seq: return None

    # 1. Interpolate Temporal
    data = np.array(valid_seq) # (Len, Features)
    current_len = len(data)

    if current_len != target_len:
        x_old = np.linspace(0, 1, current_len)
        x_new = np.linspace(0, 1, target_len)
        f = interp1d(x_old, data, axis=0, kind='linear', fill_value="extrapolate")
        data = f(x_new) # (60, Features)

    # 2. Spatial Normalization
    # Reshape (60, 67, 3)
    data_3d = data.reshape(target_len, N_TOTAL_LANDMARKS, 3)

    # Tìm bounding box toàn sequence (bỏ qua điểm 0)
    valid_mask = data_3d[:, :, 0] != 0
    if not np.any(valid_mask): return None

    all_x = data_3d[:, :, 0][valid_mask]
    all_y = data_3d[:, :, 1][valid_mask]

    min_x, max_x = np.min(all_x), np.max(all_x)
    min_y, max_y = np.min(all_y), np.max(all_y)

    width = max_x - min_x
    height = max_y - min_y
    scale = max(width, height)
    if scale < 1e-5: scale = 1 # Tránh chia 0

    # Center & Scale
    # Đưa về giữa (0.5, 0.5)
    center_x_box = min_x + width/2
    center_y_box = min_y + height/2

    # Chuẩn hóa X, Y (giữ tỉ lệ aspect ratio)
    data_3d[:, :, 0] = (data_3d[:, :, 0] - center_x_box) / scale + 0.5
    data_3d[:, :, 1] = (data_3d[:, :, 1] - center_y_box) / scale + 0.5

    # Z scale tương đối
    data_3d[:, :, 2] = data_3d[:, :, 2] / scale

    # Mask lại các điểm gốc là 0 về 0 (để tránh nhiễu nền)
    # Tuy nhiên, sau interpolate thì điểm 0 có thể bị smudge.
    # Với augmentation Keypoint, ta chấp nhận input đã clean.

    return data_3d.reshape(target_len, -1).astype(np.float32)

def generate_augmented_data(original_frames, num_augments=50):
    """Tạo ra danh sách các sequence đã augment."""
    augmented_list = []

    # Luôn thêm bản gốc
    processed_orig = interpolate_and_normalize(original_frames)
    if processed_orig is not None:
        augmented_list.append(processed_orig)

    # Danh sách hàm aug
    aug_funcs = [
        scale_keypoints_sequence,
        rotate_keypoints_sequence,
        translate_keypoints_sequence,
        time_stretch_keypoints_sequence,
        inter_hand_distance,
        add_gaussian_noise,
        stretch_compress_keypoints,
        # random_dropout # Có thể thêm dropout nếu muốn dữ liệu khó hơn
    ]

    for _ in range(num_augments):
        # Tạo bản sao
        current_seq = [x.copy() for x in original_frames]

        # Chọn ngẫu nhiên 1 đến MAX_AUGS_COMBINED kĩ thuật
        k = random.randint(1, MAX_AUGS_COMBINED)
        chosen_funcs = random.sample(aug_funcs, k)

        # Áp dụng lần lượt
        for func in chosen_funcs:
            current_seq = func(current_seq)

        # Tiền xử lý cuối cùng (Resize & Normalize)
        final_seq = interpolate_and_normalize(current_seq)
        if final_seq is not None:
            augmented_list.append(final_seq)

    return np.array(augmented_list)

# =============================================================================
# 6. MAIN EXECUTION LOOP
# =============================================================================

def main():
    print(f"Loading labels from {DATASET_PATH}...")
    label_file = os.path.join(DATASET_PATH, 'Text', 'label.csv')
    df = pd.read_csv(label_file)

    selected_actions = sorted(df['LABEL'].unique())
    label_map = {action: idx for idx, action in enumerate(selected_actions)}

    # Lưu label map
    with open(os.path.join(LOG_PATH, 'label_map.json'), 'w', encoding='utf-8') as f:
        json.dump(label_map, f, ensure_ascii=False, indent=4)

    print(f"Found {len(selected_actions)} actions.")
    print(f"Start processing... Generating {NUM_AUGMENTS_PER_VIDEO} variations per video.")

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing Videos"):
            action = row['LABEL']
            video_file = row['VIDEO']
            label_idx = label_map[action]

            # File save path: action_videoname.npz
            video_name_no_ext = os.path.splitext(video_file)[0]
            save_path = os.path.join(OUTPUT_DATA_PATH, f"{action}_{video_name_no_ext}.npz")

            # Nếu file đã tồn tại, bỏ qua (để có thể resume nếu ngắt quãng)
            if os.path.exists(save_path):
                continue

            video_path = os.path.join(DATASET_PATH, 'Videos', video_file)
            if not os.path.exists(video_path):
                # print(f"Missing video: {video_file}")
                continue

            # 1. Extract Raw Keypoints
            raw_frames = sequence_frames(video_path, holistic)
            if not raw_frames:
                continue

            # 2. Generate Augmentations (Offline)
            # Kết quả X_data sẽ có shape (N_Samples, 60, 201)
            X_data = generate_augmented_data(raw_frames, num_augments=NUM_AUGMENTS_PER_VIDEO)

            if len(X_data) == 0: continue

            # Tạo nhãn tương ứng
            y_data = np.full(len(X_data), label_idx, dtype=np.int32)

            # 3. Save Compressed
            np.savez_compressed(save_path, X=X_data, y=y_data)

    print("\nALL DATA PROCESSED AND SAVED.")
    print(f"Data saved to: {OUTPUT_DATA_PATH}")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


Loading labels from /content/drive/MyDrive/VSL_Data/Phuong/Dataset...
Found 2877 actions.
Start processing... Generating 100 variations per video.


Processing Videos:   1%|▏         | 53/3694 [13:39<15:38:49, 15.47s/it]


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/VSL_Data/Phuong/Processed_Data_Augmented_V2/ngày Quốc tế phụ nữ 8/3_D0042B.npz'

In [ ]:
import numpy as np
import os

# Thay đường dẫn tới 1 file .npz bất kỳ bạn vừa tạo
file_path = '/content/drive/MyDrive/VSL_Data/Phuong/Processed_Data_Augmented_V2/địa chỉ_D0001B.npz'

if os.path.exists(file_path):
    data = np.load(file_path)
    X = data['X']
    y = data['y']

    print(f"File: {os.path.basename(file_path)}")
    print(f"Shape của X: {X.shape}")
    # Kết quả sẽ ra dạng: (101, 60, 201)
    # -> 101: Tổng số mẫu (Gồm 1 gốc + 100 bản augment) -> ĐÃ CÓ ĐỦ Ở ĐÂY!
    # -> 60: Số frames
    # -> 201: Số điểm keypoints (x,y,z)

    print(f"Shape của y: {y.shape}")
    # Kết quả: (101,) -> 101 nhãn tương ứng
else:
    print("Chưa tìm thấy file, hãy chạy code xử lý dữ liệu trước đã nhé!")

File: địa chỉ_D0001B.npz
Shape của X: (101, 60, 201)
Shape của y: (101,)


In [6]:
import cv2
import numpy as np
import math

# --- CẤU HÌNH ---
N_UPPER_BODY = 25
N_HAND = 21
N_TOTAL = 67
# Định nghĩa đường nối (Giữ nguyên như trước)
POSE_CONN = [(11, 12), (11, 13), (13, 15), (12, 14), (14, 16), (11, 23), (12, 24), (23, 24), (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8)]
HAND_CONN = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (0, 9), (9, 10), (10, 11), (11, 12), (0, 13), (13, 14), (14, 15), (15, 16), (0, 17), (17, 18), (18, 19), (19, 20), (5, 9), (9, 13), (13, 17)]

def draw_single_pose(canvas, points, width, height, color_pose, color_hand):
    """Hàm vẽ 1 người lên canvas nhỏ"""
    # Helper vẽ đường
    def draw_lines(pts, conns, col):
        pixel_pts = []
        for p in pts:
            if p[0] == 0 and p[1] == 0: pixel_pts.append(None)
            else: pixel_pts.append((int(p[0]*width), int(p[1]*height)))

        for i, j in conns:
            if i < len(pixel_pts) and j < len(pixel_pts):
                if pixel_pts[i] and pixel_pts[j]:
                    cv2.line(canvas, pixel_pts[i], pixel_pts[j], col, 1)

    # Tách bộ phận
    pose = points[0:N_UPPER_BODY]
    lh = points[N_UPPER_BODY : N_UPPER_BODY+N_HAND]
    rh = points[N_UPPER_BODY+N_HAND :]

    draw_lines(pose, POSE_CONN, color_pose)
    draw_lines(lh, HAND_CONN, color_hand)
    draw_lines(rh, HAND_CONN, color_hand)


def create_grid_video(npz_path, output_path, grid_size=4, cell_size=200, fps=10):
    """
    grid_size: 4 nghĩa là lưới 4x4 (xem 16 mẫu cùng lúc)
    cell_size: Kích thước pixel mỗi ô nhỏ
    """
    try:
        data = np.load(npz_path)
        X = data['X'] # Shape (101, 60, 201)
    except: print("Lỗi đọc file"); return

    num_samples = min(len(X), grid_size * grid_size) # Lấy tối đa số ô lưới
    num_frames = X.shape[1] # 60 frames

    # Tính kích thước video lớn
    vid_w = grid_size * cell_size
    vid_h = grid_size * cell_size

    fourcc = cv2.VideoWriter_fourcc(*'DIVX')
    out = cv2.VideoWriter(output_path, fourcc, fps, (vid_w, vid_h))

    print(f"Đang tạo Grid Video {grid_size}x{grid_size} ({num_samples} mẫu)...")

    # Duyệt từng frame thời gian (0 -> 59)
    for t in range(num_frames):
        # Tạo khung hình lớn đen xì
        big_canvas = np.zeros((vid_h, vid_w, 3), dtype=np.uint8)

        # Duyệt từng ô trong lưới để vẽ các mẫu khác nhau
        for i in range(num_samples):
            # Tính toạ độ ô lưới
            row = i // grid_size
            col = i % grid_size

            # Lấy dữ liệu mẫu thứ i tại thời điểm t
            sample_data = X[i][t]
            points = sample_data.reshape(-1, 3)

            # Tạo canvas nhỏ cho ô này
            cell_canvas = np.zeros((cell_size, cell_size, 3), dtype=np.uint8)

            # Màu sắc: Gốc (i=0) màu xanh lá, Augment màu xanh dương/đỏ
            c_pose = (0, 255, 0) if i == 0 else (200, 200, 0) # Gốc xanh lá, Augment xanh lơ
            c_hand = (0, 0, 255) if i == 0 else (200, 50, 200)

            # Vẽ người lên ô nhỏ
            draw_single_pose(cell_canvas, points, cell_size, cell_size, c_pose, c_hand)

            # Viết chữ label (Gốc / Augment)
            label = "ORIGINAL" if i == 0 else f"Aug {i}"
            cv2.putText(cell_canvas, label, (5, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

            # Vẽ khung viền cho ô
            cv2.rectangle(cell_canvas, (0,0), (cell_size-1, cell_size-1), (50, 50, 50), 1)

            # Dán ô nhỏ vào khung hình lớn
            y_start = row * cell_size
            x_start = col * cell_size
            big_canvas[y_start : y_start+cell_size, x_start : x_start+cell_size] = cell_canvas

        out.write(big_canvas)

    out.release()
    print(f"Xong! Video lưới đã lưu tại: {output_path}")

# ==========================================
# CHẠY THỬ
# ==========================================
# Chọn file của bạn
file_input = '/content/drive/MyDrive/VSL_Data/Phuong/Processed_Data_Augmented_V2/Đu Bai (nước Đu Bai)_D0011.npz'

# Xuất video lưới 4x4 (Xem 16 biến thể cùng lúc)
create_grid_video(file_input, 'grid_view_4x4.avi', grid_size=7, cell_size=100)

# Nếu muốn xem nhiều hơn, ví dụ 5x5 (25 biến thể)
# create_grid_video(file_input, 'grid_view_5x5.avi', grid_size=5, cell_size=200)

Đang tạo Grid Video 7x7 (49 mẫu)...
Xong! Video lưới đã lưu tại: grid_view_4x4.avi
